In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import sys, json, time, tempfile
import numpy as np
import torch
import cv2
import trimesh
from pathlib import Path
from PIL import Image
import nvdiffrast.torch as dr

In [2]:
PROJECT_ROOT = Path(".").resolve().parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from objects_config import OBJECTS, gt_pose_to_matrix, rotation_error_deg, \
                           position_error_cm, add_score

FP_ROOT = Path.home() / "rai_workspace/FoundationPose"
sys.path.insert(0, str(FP_ROOT))
sys.path.insert(0, str(FP_ROOT / "mycuda"))

GD_CONFIG  = Path.home() / "rai_workspace/grounding_dino_weights/GroundingDINO_SwinT_OGC.py"
GD_WEIGHTS = Path.home() / "rai_workspace/grounding_dino_weights/groundingdino_swint_ogc.pth"
CAM_INFO   = PROJECT_ROOT / "outputs/camera_info.json"
OUT_DIR    = PROJECT_ROOT / "outputs/ablation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAM3_SCORE_THRESH = 0.10
SAM3_MASK_THRESH  = 0.50
GD_BOX_THRESHOLD  = 0.25
GD_TEXT_THRESHOLD = 0.20
MAX_PER_CAMERA    = 2
MAX_TOTAL         = 8
CLUSTER_DIST_M    = 0.30
N_FP_ITER         = 5
DEVICE            = "cuda"

In [3]:
def boxes_cxcywh_to_xyxy(boxes_norm, W, H):
    cx, cy, w, h = boxes_norm.unbind(-1)
    return torch.stack([(cx-w/2)*W, (cy-h/2)*H,
                        (cx+w/2)*W, (cy+h/2)*H], dim=-1)


def cluster_estimates(estimates, dist_m=CLUSTER_DIST_M):
    clusters = []
    for est in estimates:
        placed = False
        for cl in clusters:
            if np.linalg.norm(est["position"] - cl["rep"]["position"]) < dist_m:
                cl["members"].append(est)
                if est["score"] > cl["rep"]["score"]:
                    cl["rep"] = est
                placed = True
                break
        if not placed:
            clusters.append({"rep": est, "members": [est]})
    clusters.sort(key=lambda c: c["rep"]["score"], reverse=True)
    return clusters

In [4]:
print("Loading SAM3 …")
from transformers import Sam3Processor, Sam3Model
sam3_processor = Sam3Processor.from_pretrained("facebook/sam3")
sam3_model     = Sam3Model.from_pretrained("facebook/sam3").to(DEVICE)
sam3_model.eval()

gd_model = None

Loading SAM3 …


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

In [5]:
def load_gdino():
    global gd_model
    if gd_model is None:
        print("  [fallback] Loading GroundingDINO …")
        from groundingdino.util.inference import load_model
        gd_model = load_model(str(GD_CONFIG), str(GD_WEIGHTS))
        gd_model.eval()
    return gd_model

In [6]:
def sam3_text_detect(image_pil, prompt):
    inputs = sam3_processor(
        images=image_pil, text=prompt, return_tensors="pt"
    ).to(DEVICE)
    with torch.no_grad():
        outputs = sam3_model(**inputs)
    results = sam3_processor.post_process_instance_segmentation(
        outputs,
        threshold      = SAM3_SCORE_THRESH,
        mask_threshold = SAM3_MASK_THRESH,
        target_sizes   = inputs.get("original_sizes").tolist(),
    )[0]
    dets = []
    for mask_t, score_t, box_t in zip(
        results["masks"], results["scores"], results["boxes"]
    ):
        mask = mask_t.cpu().numpy().astype(bool)
        if not mask.any():
            continue
        dets.append({
            "mask":     mask,
            "score":    float(score_t),
            "box_xyxy": box_t.cpu().numpy(),
            "source":   "sam3_text",
        })
    return dets

In [7]:
def gdino_sam3_box_detect(image_pil, prompt, W, H):
    from groundingdino.util.inference import load_image, predict
    import os

    model = load_gdino()
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        tmp_path = tmp.name
    image_pil.save(tmp_path)
    try:
        _, img_tensor = load_image(tmp_path)
        boxes_norm, logits, _ = predict(
            model=model, image=img_tensor, caption=prompt,
            box_threshold=GD_BOX_THRESHOLD, text_threshold=GD_TEXT_THRESHOLD,
        )
    finally:
        os.unlink(tmp_path)

    if len(logits) == 0:
        return []

    boxes_xyxy = boxes_cxcywh_to_xyxy(boxes_norm, W, H)
    order      = logits.argsort(descending=True)[:MAX_PER_CAMERA]
    dets = []
    for i in order:
        box   = boxes_xyxy[i].numpy()
        inputs = sam3_processor(
            images=image_pil, input_boxes=[[box.tolist()]],
            input_boxes_labels=[[1]], return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            outputs = sam3_model(**inputs)
        results = sam3_processor.post_process_instance_segmentation(
            outputs, threshold=0.0, mask_threshold=SAM3_MASK_THRESH,
            target_sizes=inputs.get("original_sizes").tolist(),
        )[0]
        if len(results["masks"]) == 0:
            continue
        mask = results["masks"][0].cpu().numpy().astype(bool)
        if not mask.any():
            continue
        dets.append({
            "mask":     mask,
            "score":    logits[i].item(),
            "box_xyxy": box,
            "source":   "gdino_sam3_box",
        })
    return dets

In [8]:
print("Loading FoundationPose scorer + refiner …")
from estimater import FoundationPose, PoseRefinePredictor, ScorePredictor
scorer  = ScorePredictor()
refiner = PoseRefinePredictor()
glctx   = dr.RasterizeCudaContext()

with open(CAM_INFO) as f:
    camera_info = json.load(f)

Loading FoundationPose scorer + refiner …
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


[__init__()] self.cfg: 
 lr: 0.0001
c_in: 6
zfar: 'Infinity'
debug: null
n_view: 1
run_id: 3wy8qqex
use_BN: true
exp_name: 2024-01-11-20-02-45
n_epochs: 62
save_dir: /home/bowenw/debug/2024-01-11-20-02-45/
use_mask: false
loss_type: pairwise_valid
optimizer: adam
batch_size: 64
crop_ratio: 1.1
enable_amp: true
use_normal: false
max_num_key: null
warmup_step: -1
input_resize:
- 160
- 160
max_step_val: 1000
vis_interval: 1000
weight_decay: 0
normalize_xyz: true
resume_run_id: null
clip_grad_norm: 'Infinity'
lr_epoch_decay: 500
render_backend: nvdiffrast
train_num_pair: 5
lr_decay_epochs:
- 50
n_epochs_warmup: 1
make_pair_online: false
gradient_max_norm: 'Infinity'
max_step_per_epoch: 10000
n_rendering_workers: 1
save_epoch_interval: 100
n_dataloader_workers: 100
split_objects_across_gpus: true
ckpt_dir: /home/salman/rai_workspace/FoundationPose/learning/training/../../weights/2024-01-11-20-02-45/model_best.pth

[__init__()] self.h5_file:None


Warp 1.13.0 initialized:
   CUDA Toolkit 12.9, Driver 12.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA RTX 6000 Ada Generation" (48 GiB, sm_89, mempool enabled)
   Kernel cache:
     /home/salman/.cache/warp/1.13.0


[__init__()] Using pretrained model from /home/salman/rai_workspace/FoundationPose/learning/training/../../weights/2024-01-11-20-02-45/model_best.pth
/home/salman/rai_workspace/FoundationPose/learning/training/predict_score.py:151: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control o

In [9]:
results = {}

for label, cfg in OBJECTS.items():
    print(f"\n{'='*60}")
    print(f"Object: {label}  prompt: '{cfg['prompt']}'")
    print(f"{'='*60}")

    T_gt   = gt_pose_to_matrix(cfg["gt_pose"])
    result = {"status": "no_detection"}

    t0_det   = time.perf_counter()
    all_dets = []

    for cam_name, info in camera_info.items():
        W, H      = info["width"], info["height"]
        image_pil = Image.open(info["rgb_path"]).convert("RGB")

        dets = sam3_text_detect(image_pil, cfg["prompt"])
        if not dets:
            dets = gdino_sam3_box_detect(image_pil, cfg["prompt"], W, H)

        dets.sort(key=lambda d: d["score"], reverse=True)
        for det in dets[:MAX_PER_CAMERA]:
            det["cam"]  = cam_name
            det["info"] = info
            all_dets.append(det)

    sam3_model.to("cpu")
    torch.cuda.empty_cache()

    det_time = time.perf_counter() - t0_det

    if not all_dets:
        print(f"  No detections — skipping")
        sam3_model.to(DEVICE)   # restore for next object
        results[label] = result
        continue

    all_dets.sort(key=lambda d: d["score"], reverse=True)
    top_dets = all_dets[:MAX_TOTAL]
    print(f"  {len(top_dets)} detection(s) in {det_time:.2f}s  "
          f"(sources: {set(d['source'] for d in top_dets)})")
    
    dbg_dir = OUT_DIR / "debug" / f"{label}_sam3"
    dbg_dir.mkdir(parents=True, exist_ok=True)
    for di, det in enumerate(top_dets):
        rgb_d   = cv2.imread(det["info"]["rgb_path"])
        mask_u8 = det["mask"].astype(np.uint8) * 255
        overlay = np.zeros_like(rgb_d); overlay[:] = (255, 100, 0)
        rgb_d   = cv2.addWeighted(rgb_d, 1.0,
                      cv2.bitwise_and(overlay, overlay, mask=mask_u8), 0.4, 0)
        x1, y1, x2, y2 = det["box_xyxy"].astype(int)
        cv2.rectangle(rgb_d, (x1, y1), (x2, y2), (255, 100, 0), 3)
        cv2.putText(rgb_d,
                    f"{det['cam']}  s={det['score']:.2f}  [{det['source']}]",
                    (x1, max(y1 - 8, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 100, 0), 2)
        cv2.imwrite(str(dbg_dir / f"det_{di:02d}_{det['cam']}.png"), rgb_d)
    print(f"  Det viz saved → {dbg_dir}")

    mesh_path = PROJECT_ROOT / "megapose_objects" / label / "mesh.ply"
    mesh      = trimesh.load(str(mesh_path))
    fp_est    = FoundationPose(
        model_pts     = np.array(mesh.vertices,       dtype=np.float32),
        model_normals = np.array(mesh.vertex_normals,  dtype=np.float32),
        mesh          = mesh,
        scorer        = scorer,
        refiner       = refiner,
        debug_dir     = "/tmp/fp_debug",
        debug         = 0,
        glctx         = glctx,
    )

    verts_hom = np.hstack([np.array(mesh.vertices),
                            np.ones((len(mesh.vertices), 1))])

    t0_est    = time.perf_counter()
    estimates = []

    for det in top_dets:
        info  = det["info"]
        rgb   = np.array(Image.open(info["rgb_path"]).convert("RGB"))
        depth = np.load(info["depth_path"]).astype(np.float32)
        K     = np.array(info["K"],           dtype=np.float64)
        T_wc  = np.array(info["T_world_cam"], dtype=np.float64)

        try:
            T_co = fp_est.register(
                K=K, rgb=rgb, depth=depth,
                ob_mask=det["mask"], iteration=N_FP_ITER,
            )
        except Exception as e:
            print(f"    FP failed: {e}")
            continue

        T_wo = T_wc @ T_co


        try:
            H_v, W_v = rgb.shape[:2]
            rgb_viz  = cv2.imread(info["rgb_path"])
            verts_c  = (T_co @ verts_hom.T).T[:, :3]
            front    = verts_c[:, 2] > 0
            if front.any():
                pts2d = (K @ verts_c[front].T).T
                pts2d = (pts2d[:, :2] / pts2d[:, 2:3]).astype(int)
                ok    = ((pts2d[:, 0] >= 0) & (pts2d[:, 0] < W_v) &
                         (pts2d[:, 1] >= 0) & (pts2d[:, 1] < H_v))
                for pt in pts2d[ok][::10]:
                    cv2.circle(rgb_viz, tuple(pt), 3, (0, 165, 255), -1)
            al = 0.05
            def prj(p, K=K, T=T_co):
                q = K @ p; return (int(q[0]/q[2]), int(q[1]/q[2]))
            o = prj(T_co[:3, 3])
            cv2.arrowedLine(rgb_viz, o, prj(T_co[:3,3]+al*T_co[:3,0]), (0,0,255), 3, tipLength=0.3)
            cv2.arrowedLine(rgb_viz, o, prj(T_co[:3,3]+al*T_co[:3,1]), (0,255,0), 3, tipLength=0.3)
            cv2.arrowedLine(rgb_viz, o, prj(T_co[:3,3]+al*T_co[:3,2]), (255,0,0), 3, tipLength=0.3)
            cv2.putText(rgb_viz,
                        f"pos=[{T_wo[0,3]:.3f},{T_wo[1,3]:.3f},{T_wo[2,3]:.3f}]",
                        (10, H_v - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
            cv2.imwrite(str(dbg_dir / f"pose_{len(estimates):02d}_{det['cam']}.png"), rgb_viz)
        except Exception as ve:
            print(f"    Pose viz failed: {ve}")


        estimates.append({
            "position": T_wo[:3, 3],
            "score":    det["score"],
            "T_wo":     T_wo,
        })

    est_time = time.perf_counter() - t0_est

    sam3_model.to(DEVICE)

    if not estimates:
        print(f"  No pose estimates produced")
        result["status"] = "estimation_failed"
        results[label]   = result
        continue

    clusters = cluster_estimates(estimates)
    best     = clusters[0]["rep"]
    T_est    = best["T_wo"]

    pos_err            = position_error_cm(T_est[:3, 3], T_gt[:3, 3])
    rot_err            = rotation_error_deg(T_est[:3, :3], T_gt[:3, :3])
    add_cm, diam_cm, add_ok = add_score(np.array(mesh.vertices), T_est, T_gt)

    print(f"  Position error : {pos_err:.1f} cm")
    print(f"  Rotation error : {rot_err:.1f} deg")
    print(f"  ADD            : {add_cm:.3f} cm  (diam={diam_cm:.2f} cm)  {'✓' if add_ok else '✗'}")
    print(f"  Time           : det={det_time:.2f}s  est={est_time:.2f}s")

    results[label] = {
        "status":             "success",
        "position_error_cm":  round(pos_err, 3),
        "rotation_error_deg": round(rot_err, 3),
        "add_cm":             round(add_cm, 4),
        "add_diameter_cm":    round(diam_cm, 4),
        "add_success":        bool(add_ok),
        "detection_time_s":   round(det_time, 3),
        "estimation_time_s":  round(est_time, 3),
        "total_time_s":       round(det_time + est_time, 3),
        "n_detections_used":  len(estimates),
        "T_est":              T_est.tolist(),
        "T_gt":               T_gt.tolist(),
    }


Object: lamp  prompt: 'yellow lamp'


  [fallback] Loading GroundingDINO …
final text_encoder_type: bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


  8 detection(s) in 7.63s  (sources: {'sam3_text', 'gdino_sam3_box'})
  Det viz saved → /home/salman/rai_workspace/ablation_study/outputs/ablation/debug/lamp_sam3


...

    FP failed: CUDA out of memory. Tried to allocate 10.38 GiB. GPU 0 has a total capacity of 47.51 GiB of which 1.87 GiB is free. Process 770410 has 23.94 GiB memory in use. Including non-PyTorch memory, this process has 21.66 GiB memory in use. Of the allocated memory 8.66 GiB is allocated by PyTorch, and 6.98 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_cr

    FP failed: CUDA out of memory. Tried to allocate 10.38 GiB. GPU 0 has a total capacity of 47.51 GiB of which 1.87 GiB is free. Process 770410 has 23.94 GiB memory in use. Including non-PyTorch memory, this process has 21.66 GiB memory in use. Of the allocated memory 8.66 GiB is allocated by PyTorch, and 6.98 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


[predict()] ob_in_cams:(252, 4, 4)
[predict()] self.cfg.use_normal:False
[predict()] trans_normalizer:[0.019999999552965164, 0.019999999552965164, 0.05000000074505806], rot_normalizer:0.3490658503988659
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()

    FP failed: CUDA out of memory. Tried to allocate 10.38 GiB. GPU 0 has a total capacity of 47.51 GiB of which 1.87 GiB is free. Process 770410 has 23.94 GiB memory in use. Including non-PyTorch memory, this process has 21.66 GiB memory in use. Of the allocated memory 8.66 GiB is allocated by PyTorch, and 6.98 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
  No pose estimates produced


In [ ]:
output = {"method": "fp_sam3", "results": results}
out_path = OUT_DIR / "results_fp_sam3.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"\nSaved → {out_path}")